In [ ]:
import boto3
from datetime import datetime, timedelta, timezone
from botocore.exceptions import ClientError

BUCKET_NAME = 'bucket_name' # Reemplaza con tu bucket real
DIAS_DE_RETENCION = 30

# Inicializar cliente
s3_client = boto3.client('s3')

In [2]:
def cleanup_old_backups(bucket_name, retention_days):
    """
    Lista y elimina objetos en S3 que tengan más de 'retention_days' de antigüedad.
    """
    ahora = datetime.now(timezone.utc)
    limite_fecha = ahora - timedelta(days=retention_days)
    
    print(f"Buscando archivos anteriores a: {limite_fecha.strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"Bucket: {bucket_name}\n")

    try:
        # Listar objetos en el bucket
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket_name)

        objetos_eliminados = 0
        
        for page in pages:
            if 'Contents' in page:
                for obj in page['Contents']:
                    fecha_archivo = obj['LastModified']
                    nombre_archivo = obj['Key']
                    
                    # Verificar si el archivo es más antiguo que el límite
                    if fecha_archivo < limite_fecha:
                        print(f"Eliminando: {nombre_archivo} (Fecha: {fecha_archivo.strftime('%Y-%m-%d')})")
                        
                        s3_client.delete_object(Bucket=bucket_name, Key=nombre_archivo)
                        objetos_eliminados += 1
            else:
                print("El bucket está vacío o no tiene objetos que coincidan.")

        print(f"\nLimpieza completada. Total de objetos eliminados: {objetos_eliminados}")

    except ClientError as e:
        print(f"Error al conectar con S3: {e}")
    except Exception as e:
        print(f"Error inesperado: {e}")

# --- EJECUCIÓN ---
cleanup_old_backups(BUCKET_NAME, DIAS_DE_RETENCION)

Buscando archivos anteriores a: 2026-03-14 22:31:48 UTC
Bucket: s3-sync-project-vv


Limpieza completada. Total de objetos eliminados: 0
